# 07 - Mazes (T3)
2D backbone (r = 4 kernel-3 convolutions per step), trained on 9x9 mazes, tested on 13x13 to
33x33. Depth proxies: dead-end-filling depth (used for the lower-bound column, so "premature" here
means halting before a local dead-end filler could finish) and optimal path length. There is no
tight lower bound for mazes, so this task is empirical only.

In [ ]:
# parameters
CONFIGS = ["configs/main_mazes.yaml"]
TASKS = ["mazes"]
SPLITS = ["id_test", "ood_13", "ood_17", "ood_21", "ood_25", "ood_33"]

In [ ]:
%matplotlib inline
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from glwm2 import PROJECT_ROOT, RESULTS_DIR, PAPER_DIR
from glwm2.eval.plotting import set_style
set_style()
WRITTEN = []  # files this notebook writes

In [ ]:
from glwm2.eval.results import configs_from_yaml, finished, curve_rows, step_acc
from glwm2.eval.analysis import (ENTRIES, BASELINES, OURS, POOLED_JOINT, CONF_RULES, entry_rows, entry_instances,
                                 summary_table, g2_check, g3_check, binned, premature_ci, length_leakage_ci)
from glwm2.eval.plotting import savefig, FAMILY_COLORS
from glwm2.utils.io import write_json
cfgs = {t: [c for f in CONFIGS for c in configs_from_yaml(PROJECT_ROOT / f) if c.task == t] for t in TASKS}
for t in TASKS:
    print(t, f"{len(finished(cfgs[t]))}/{len(cfgs[t])} runs finished, seeds:",
          sorted({c.seed for c in finished(cfgs[t])}))
rows = {t: entry_rows(cfgs[t]) for t in TASKS}

In [ ]:
r = rows["mazes"]
if len(r):
    display(summary_table(r, SPLITS, "acc")["mean"].round(3))
    display(summary_table(r, SPLITS, "mean_steps")["mean"].round(1))
    display(summary_table(r, SPLITS, "premature")["mean"].round(3))

In [ ]:
show = ["oracle_stable", "fixed_T", "max_softmax", "stability", "kl", "act", "ponder", "trm_joint", "done_post", "done_prop_post", "done_joint"]
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.5))
if len(r):
    sizes = {"id_test": 9, "ood_13": 13, "ood_17": 17, "ood_21": 21, "ood_25": 25, "ood_33": 33}
    for e in show:
        sub = r[(r.entry == e) & r.split.isin(SPLITS)].groupby("split")[["acc", "mean_steps"]].mean()
        if sub.empty: continue
        xs = [sizes[s] for s in sub.index]
        fam = ENTRIES[e][2]
        kw = dict(marker=".", color=FAMILY_COLORS[fam], lw=1.2 if fam == "ours" else 0.7, label=ENTRIES[e][3])
        axes[0].plot(xs, sub.acc.values, **kw); axes[1].plot(xs, sub.mean_steps.values, **kw)
axes[0].set_xlabel("maze size n"); axes[0].set_ylabel("accuracy"); axes[1].set_xlabel("maze size n"); axes[1].set_ylabel("mean steps")
axes[1].set_yscale("log"); axes[0].legend(fontsize=5.5, frameon=False, ncol=2)
WRITTEN.append(savefig(fig, "fig7_mazes"))

In [ ]:
if len(r):
    inst = pd.concat([entry_instances(cfgs["mazes"], s, show) for s in SPLITS])
    print("Spearman(steps, dead-end depth) per entry:")
    print(inst.groupby("entry").apply(lambda d: d[["steps", "depth"]].corr(method="spearman").iloc[0, 1], include_groups=False).round(3).to_string())
    WRITTEN.append(write_json(RESULTS_DIR / "mazes_summary.json", r.groupby(["entry", "split"])[["acc", "mean_steps", "premature"]].mean().reset_index().to_dict(orient="records")))

In [ ]:
print("Files written by this notebook:")
for p in WRITTEN:
    print(" ", Path(p).relative_to(PROJECT_ROOT))